In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# ============================================================
# BlockA (H2O_v4_parallel_to_stdplev)
#
# Parallel read ALL h3 files, compute:
#   - zonal mean H2O, p_mid
#   - 60–90N cos(lat)-weighted polar-cap mean on hybrid lev
#   - log-p interpolate polar-cap H2O to STANDARD pressure levels (Pa)
#
# Output (ONLY one cache):
#   H2O_HIND_pcap_60_90N_stdplev.nc   (time, plev[Pa])
#
# Standard plevs (Pa) are fixed to match your other files:
#   [1000, 5000, 7000, 10000, 15000, ..., 100000]
#
# H2O units are mol/mol in cache (convert to ppm in B/C/D).
# ============================================================

import os, glob
import numpy as np
import xarray as xr
from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm

# ================= 配置区 =================
ROOT_OUT = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_H2O"
H3_DIR   = "/mnt/backup_ETH/EXTR_2000/EXTR_2000/BWCN.e122.f19_g16.002/atm/hist"
OUT_STD_PCAP = os.path.join(ROOT_OUT, "H2O_HIND_pcap_60_90N_stdplev.nc")

# 标准 pressure levels (Pa)
PLEV_STD_PA = np.array([
    10, 50, 100, 200, 300, 500, 1000, 2000, 3000, 5000, 7000, 10000, 15000, 20000, 25000, 30000, 40000, 50000, 60000, 70000, 85000, 92500, 100000
], dtype=float)

MAX_WORKERS = 116
# =========================================

def compute_pressure_mid(ds):
    """p_mid = hyam*P0 + hybm*PS  (Pa)"""
    hyam = ds["hyam"]   # (lev)
    hybm = ds["hybm"]
    P0   = ds["P0"]     # scalar Pa
    PS   = ds["PS"]     # (time, lat, lon) Pa
    p_mid = hyam * P0 + hybm * PS
    p_mid.name = "p_mid"
    p_mid.attrs["units"] = "Pa"
    return p_mid

def lat_weighted_mean(da, lat_name="lat"):
    """cos(lat)-weighted mean over latitude."""
    lat = da[lat_name]
    w = np.cos(np.deg2rad(lat))
    w = w / w.mean()
    return da.weighted(w).mean(lat_name)

def interp_profile_logp(v_hyb, p_hyb, p_tgt):
    """
    v_hyb, p_hyb: (..., lev) with p_hyb in Pa.
    p_tgt: (nplev,) target Pa.
    Return (..., nplev) log-p interpolated.
    """
    p_tgt = np.asarray(p_tgt, float)

    def _interp_col(vcol, pcol):
        m = np.isfinite(vcol) & np.isfinite(pcol) & (pcol > 0)
        if m.sum() < 2:
            return np.full(p_tgt.shape, np.nan, float)
        p_use = pcol[m]
        v_use = vcol[m]
        # sort by pressure ascending for np.interp
        idx = np.argsort(p_use)
        p_use = p_use[idx]
        v_use = v_use[idx]
        return np.interp(np.log(p_tgt), np.log(p_use), v_use,
                         left=np.nan, right=np.nan)

    out = xr.apply_ufunc(
        _interp_col,
        v_hyb, p_hyb,
        input_core_dims=[["lev"], ["lev"]],
        output_core_dims=[["plev"]],
        vectorize=True,
        dask="parallelized",
        output_dtypes=[float],
    )
    out = out.assign_coords(plev=("plev", p_tgt))
    out["plev"].attrs["units"] = "Pa"
    return out

def process_single_file(fp):
    """
    Process one h3 file -> return DataArray (time, plev_std)
    already polar-cap weighted and interpolated.
    """
    try:
        with xr.open_dataset(fp, use_cftime=True) as ds:
            if "H2O" not in ds:
                return None

            h2o = ds["H2O"].load()             # (time, lev, lat, lon)
            p_mid = compute_pressure_mid(ds).load()

            # ZM
            h2o_zm = h2o.mean("lon")           # (time, lev, lat)
            p_zm   = p_mid.mean("lon")        # (time, lev, lat)

            # polar cap mean on hybrid lev
            h2o_pc_hyb = lat_weighted_mean(h2o_zm.sel(lat=slice(60,90)), "lat")  # (time, lev)
            p_pc_hyb   = lat_weighted_mean(p_zm.sel(lat=slice(60,90)), "lat")    # (time, lev)

            # log-p interpolate to standard plevs
            h2o_pc_std = interp_profile_logp(h2o_pc_hyb, p_pc_hyb, PLEV_STD_PA)  # (time, plev)

            # quick sanity: if too many NaNs, warn
            nan_frac = np.isnan(h2o_pc_std.values).mean()
            if nan_frac > 0.2:
                print(f"⚠️ High NaN fraction ({nan_frac:.2f}) in {os.path.basename(fp)}")

            return h2o_pc_std

    except Exception as e:
        print(f"\n[Error] Processing {os.path.basename(fp)} failed: {e}")
        return None

def main_blockA_H2O_v4_parallel_to_stdplev():
    os.makedirs(ROOT_OUT, exist_ok=True)
    h3_files = sorted(glob.glob(os.path.join(H3_DIR, "*.cam.h3.*.nc")))
    total_files = len(h3_files)
    print(f"[BlockA_H2O_v4] Found {total_files} h3 files. Parallel processing...")

    pieces = []
    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
        results = list(tqdm(executor.map(process_single_file, h3_files),
                            total=total_files))

    for r in results:
        if r is not None:
            pieces.append(r)

    if len(pieces) == 0:
        raise RuntimeError("No valid H2O data were found in h3 files.")

    print("[BlockA_H2O_v4] Concatenating...")
    da_std = xr.concat(pieces, dim="time").sortby("time").transpose("time","plev")

    da_std.name = "H2O_HIND_pcap_60_90N_stdplev"
    da_std.attrs.update(
        units="mol/mol",
        long_name="H2O polar-cap (60–90N cos-weighted) interpolated to standard pressure levels"
    )

    da_std.to_netcdf(OUT_STD_PCAP)
    print("[BlockA_H2O_v4] Saved:", OUT_STD_PCAP)
    print("[BlockA_H2O_v4] Done.")

if __name__ == "__main__":
    main_blockA_H2O_v4_parallel_to_stdplev()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# ============================================================
# BlockB (H2O_v4_1_ppm_stdplev_plot)
#
# Fix legend parsing issue by passing handles+labels explicitly.
# ============================================================

import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib import rcParams

ROOT_H2O = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_H2O"
IN_STD = os.path.join(ROOT_H2O, "H2O_HIND_pcap_60_90N_stdplev.nc")

N_PREV = 91
YEARS_SPECIAL = np.array([8, 13, 14, 19], dtype=int)

PLEV_PLOT_PA = [1000, 5000, 10000]  # Pa = 10/50/100 hPa

XTICKS = [0, 31, 61, 91, 122, 150, 181, 211, 242, 273, 304, 334]
XTICK_LABELS = ["Oct","Nov","Dec","Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep"]

def _select_year(da, year):
    return da.sel(time=da.time.dt.year == year)

def stitch_prev_oct_to_curr_sep(da_1lev, year):
    prev = _select_year(da_1lev, year-1)
    curr = _select_year(da_1lev, year)
    if prev.sizes["time"] == 0 or curr.sizes["time"] == 0:
        return xr.zeros_like(da_1lev.isel(time=slice(0,0)))
    prev_oct_dec = prev.sel(time=prev.time.dt.month.isin([10,11,12]))
    curr_jan_sep = curr.sel(time=curr.time.dt.month.isin([1,2,3,4,5,6,7,8,9]))
    return xr.concat([prev_oct_dec, curr_jan_sep], dim="time")

def build_allyears_climatology(var_1lev):
    n_days = int(var_1lev.time.dt.dayofyear.max())
    clim_mean = var_1lev.groupby("time.dayofyear").mean("time")
    clim_std  = var_1lev.groupby("time.dayofyear").std("time")
    clim_mean = np.array(clim_mean); clim_std = np.array(clim_std)

    prev_mean = clim_mean[n_days-N_PREV:n_days]
    prev_std  = clim_std[n_days-N_PREV:n_days]
    cur_mean  = clim_mean[0:n_days-N_PREV]
    cur_std   = clim_std[0:n_days-N_PREV]

    comp_mean = np.concatenate([prev_mean, cur_mean])
    comp_std  = np.concatenate([prev_std,  cur_std])
    return comp_mean, comp_std, n_days

def plot_one_plev(var_ppm_1lev, plev_pa, out_png, out_pdf):
    all_mean, all_std, n_days = build_allyears_climatology(var_ppm_1lev)
    x_full = np.arange(n_days)
    years_all = np.unique(var_ppm_1lev.time.dt.year.values.astype(int))

    rcParams.update({
        "font.family":"sans-serif",
        "font.sans-serif":["Arial","DejaVu Sans"],
        "font.size":9,
        "axes.spines.top":False,
        "axes.spines.right":False,
    })

    fig, ax = plt.subplots(1,1,figsize=(6.5,4.0), constrained_layout=True)

    colors_spec = plt.cm.tab10(np.linspace(0,1,len(YEARS_SPECIAL)))
    handles_years = []  # make sure it's a pure python list

    for i,y in enumerate(YEARS_SPECIAL):
        if y not in years_all:
            continue
        stitched = stitch_prev_oct_to_curr_sep(var_ppm_1lev, y)
        if stitched.sizes["time"] != n_days:
            continue
        ax.plot(x_full, stitched.values, color=colors_spec[i], lw=1.5)
        handles_years.append(
            Line2D([0],[0], color=colors_spec[i], lw=1.5, label=f"Year {y:04d}")
        )

    ax.fill_between(x_full, all_mean-all_std, all_mean+all_std,
                    color="0.85", alpha=0.9, linewidth=0)
    ax.plot(x_full, all_mean, color="black", lw=1.8, ls="--")

    ax.set_xlim(0,n_days)
    ax.set_xticks(XTICKS); ax.set_xticklabels(XTICK_LABELS)
    ax.set_xlabel("Seasonal months (Oct–Sep)")
    ax.set_ylabel("H2O mixing ratio (60–90°N weighted mean) [ppm]")
    ax.set_title(f"H2O polar cap 60–90°N — {plev_pa/100:.0f} hPa")

    patch_all = Patch(facecolor="0.85", edgecolor="none", label="Climatology ±1σ")
    line_all  = Line2D([0],[0], color="black", ls="--", lw=1.8, label="All-years climatology")

    handles = handles_years + [line_all, patch_all]
    labels  = [h.get_label() for h in handles]

    # ✅ robust legend: explicit handles + labels
    ax.legend(handles=handles, labels=labels,
              loc="best", frameon=False, fontsize=8, ncol=2)

    plt.savefig(out_png, dpi=300)
    plt.savefig(out_pdf)
    plt.close(fig)
    print("[BlockB_H2O_v4.1] Saved:", out_png)

def main_blockB_H2O_v4_1_ppm_stdplev_plot():
    da = xr.load_dataarray(IN_STD)  # (time, plev Pa)
    da_ppm = da * 1e6

    for p in PLEV_PLOT_PA:
        var_1lev = da_ppm.sel(plev=p, method="nearest")
        out_png = os.path.join(ROOT_H2O, f"H2O_pcap60_90N_{p}Pa_special_vs_climatology.png")
        out_pdf = os.path.join(ROOT_H2O, f"H2O_pcap60_90N_{p}Pa_special_vs_climatology.pdf")
        plot_one_plev(var_1lev, p, out_png, out_pdf)

    print("[BlockB_H2O_v4.1] Done.")

if __name__ == "__main__":
    main_blockB_H2O_v4_1_ppm_stdplev_plot()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
BlockC (H2O_v6_vert_anom_sig_stdplev_boot_O3style_excl_special)

Fixes:
1) Significance logic aligned with O3:
   - t-test (single-observation vs baseline DOY distribution)
   - bootstrap CI (classic) on baseline DOY distribution
2) Baseline climatology and baseline anomalies EXCLUDE special years
   AND their previous years (because Oct–Dec comes from y-1).
3) Keep your Oct–Sep stitching + classic bootstrap logic.
"""

import os
import numpy as np
import xarray as xr
from scipy.stats import t as t_dist
from joblib import Parallel, delayed
import time

# ================= Config =================
ROOT_H2O = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_H2O"
IN_STD = os.path.join(ROOT_H2O, "H2O_HIND_pcap_60_90N_stdplev.nc")
OUT_C  = os.path.join(ROOT_H2O, "H2O_vert_anom_special_stdplev.nc")

YEARS_SPECIAL = np.array([8, 13, 14, 19], dtype=int)
N_PREV = 91
ALPHA = 0.05
NBOOT = 5000
RNG_SEED = 42
N_JOBS = 200
# =========================================


def _select_year(da, year):
    return da.sel(time=da.time.dt.year == year)

def stitch_prev_oct_to_curr_sep(da_plev, year):
    prev = _select_year(da_plev, year-1)
    curr = _select_year(da_plev, year)
    prev_oct_dec = prev.sel(time=prev.time.dt.month.isin([10,11,12]))
    curr_jan_sep = curr.sel(time=curr.time.dt.month.isin([1,2,3,4,5,6,7,8,9]))
    return xr.concat([prev_oct_dec, curr_jan_sep], dim="time")

def bootstrap_ci_classic(data_1d, nboot=5000, alpha=0.05, rng=None):
    data = np.asarray(data_1d, float)
    data = data[np.isfinite(data)]
    n = data.size
    if n < 2:
        return np.nan, np.nan

    if rng is None:
        rng = np.random.default_rng()

    means = np.empty(nboot, dtype=float)
    for i in range(nboot):
        samp = rng.choice(data, size=n, replace=True)
        means[i] = samp.mean()

    low  = np.percentile(means, 100 * alpha / 2)
    high = np.percentile(means, 100 * (1 - alpha / 2))
    return low, high


### [FIX] O3-style significance for one DOY column
def sig_one_day_O3style(doy, base_anom_vals, doy_base, obs_anom_1d,
                        alpha=0.05, nboot=5000, seed=0):
    """
    base_anom_vals: (time_base, plev)
    doy_base      : (time_base,)
    obs_anom_1d   : (plev,) anomaly for this special year & this doy
    returns:
        t_sig (plev,), b_sig (plev,)
    """
    n_plev = base_anom_vals.shape[1]
    t_sig = np.zeros(n_plev, dtype=bool)
    b_sig = np.zeros(n_plev, dtype=bool)

    mask_day = (doy_base == doy)
    if not np.any(mask_day):
        return t_sig, b_sig

    day_samples = base_anom_vals[mask_day, :]  # (n_samp, plev)
    rng = np.random.default_rng(seed)

    for k in range(n_plev):
        col = day_samples[:, k]
        col = col[np.isfinite(col)]
        if col.size < 2:
            continue

        obs = obs_anom_1d[k]

        # --- t-test (single obs vs baseline mean=0) ---
        se = np.std(col, ddof=1) / np.sqrt(col.size)
        if se > 0:
            tstat = obs / se
            pval = 2 * (1 - t_dist.cdf(np.abs(tstat), df=col.size - 1))
            t_sig[k] = (pval < alpha)

        # --- bootstrap CI on baseline mean ---
        lo, hi = bootstrap_ci_classic(col, nboot=nboot, alpha=alpha, rng=rng)
        if np.isfinite(lo) and np.isfinite(hi):
            b_sig[k] = not (lo <= obs <= hi)

    return t_sig, b_sig


def main_blockC_v6():
    t0 = time.time()
    print(f"[BlockC_v6] Loading data...")

    da = xr.load_dataarray(IN_STD)  # (time, plev)
    da_ppm = da * 1e6
    plev_vals = da_ppm.plev.values

    years_all = da_ppm.time.dt.year.values.astype(int)
    doy_all   = da_ppm.time.dt.dayofyear.values.astype(int)
    n_days = int(doy_all.max())
    print("[BlockC_v6] n_days =", n_days)

    # =========================================================
    # ### [FIX] Baseline exclusion
    # Exclude special years AND their previous years
    # =========================================================
    years_excl = np.unique(np.concatenate([YEARS_SPECIAL, YEARS_SPECIAL - 1]))
    print("[BlockC_v6] Excluding years from baseline/clim:", years_excl)

    base_da = da_ppm.sel(time=~da_ppm.time.dt.year.isin(years_excl))
    base_years = base_da.time.dt.year.values.astype(int)
    doy_base   = base_da.time.dt.dayofyear.values.astype(int)

    print("[BlockC_v6] Baseline years:", np.unique(base_years))

    # ===== Baseline climatology from excluded dataset only =====
    clim_base = base_da.groupby("time.dayofyear").mean("time")  # (dayofyear, plev)

    clim_base_vals = clim_base.values  # (n_days, plev)

    # stitched climatology Oct–Sep (same logic as before)
    clim_base_arr = np.array(clim_base)
    clim_prev = clim_base_arr[n_days - N_PREV : n_days, :]
    clim_cur  = clim_base_arr[0 : n_days - N_PREV, :]
    clim_stitched = np.concatenate([clim_prev, clim_cur], axis=0)  # (n_days, plev)

    # ===== baseline anomalies for significance =====
    clim_sel_for_base = clim_base.sel(dayofyear=base_da.time.dt.dayofyear)
    base_anom = base_da - clim_sel_for_base   # (time_base, plev), ppm already

    base_anom_vals = base_anom.values

    # map DOY -> indices inside baseline
    doy_map_base = {d: np.where(doy_base == d)[0] for d in np.unique(doy_base)}
    doys_full = np.arange(1, n_days + 1)

    # =========================================================
    # Special years: anomaly relative to stitched clim
    # + significance masks O3-style
    # =========================================================
    print("[BlockC_v6] Assembling special years and significance ...")
    anom_list=[]; tmask_list=[]; boot_list=[]

    for y in YEARS_SPECIAL:
        stitched = stitch_prev_oct_to_curr_sep(da_ppm, y)  # (n_days, plev)
        L = min(stitched.shape[0], clim_stitched.shape[0])

        vals = stitched.values[:L, :]
        clim = clim_stitched[:L, :]
        anom = vals - clim  # (L, plev), ppm

        anom_list.append(anom)

        # O3-style significance
        tmask = np.zeros_like(anom, dtype=bool)
        bmask = np.zeros_like(anom, dtype=bool)

        # parallel over time_index if you want (here keep simple + stable)
        for it in range(L):
            true_doy = int(stitched.time.dt.dayofyear.values[it])
            # baseline distribution for that DOY
            idxs = doy_map_base.get(true_doy, [])
            if len(idxs) == 0:
                continue

            t_sig_1d, b_sig_1d = sig_one_day_O3style(
                true_doy,
                base_anom_vals,
                doy_base,
                anom[it, :],
                alpha=ALPHA,
                nboot=NBOOT,
                seed=RNG_SEED + true_doy
            )
            tmask[it, :] = t_sig_1d
            bmask[it, :] = b_sig_1d

        tmask_list.append(tmask)
        boot_list.append(bmask)

    ds_out = xr.Dataset(
        data_vars=dict(
            anom_ppm=(["ref_year","time_index","plev"], np.stack(anom_list,0)),
            tmask=(["ref_year","time_index","plev"], np.stack(tmask_list,0)),
            bootmask=(["ref_year","time_index","plev"], np.stack(boot_list,0)),
            clim_mean_ppm=(["time_index","plev"], clim_stitched[:L]),
        ),
        coords=dict(
            ref_year=("ref_year", YEARS_SPECIAL),
            time_index=("time_index", np.arange(L)),
            plev=("plev", plev_vals),
        )
    )

    ds_out["plev"].attrs["units"]="Pa"
    ds_out["anom_ppm"].attrs["units"]="ppm"
    ds_out["clim_mean_ppm"].attrs["units"]="ppm"

    ds_out.attrs["description"] = (
        "H2O polar-cap (60–90N) time–height anomalies in ppm for special years "
        "relative to baseline climatology constructed from NON-special years. "
        "Baseline excludes special years and their previous years to avoid "
        "contamination under Oct–Sep stitching. "
        "Significance masks follow O3-style: t-test and classic bootstrap CI "
        "based on baseline DOY anomaly distributions."
    )

    ds_out.to_netcdf(OUT_C)
    print(f"[BlockC_v6] Saved: {OUT_C}")
    print(f"[BlockC_v6] Total time: {time.time() - t0:.2f} s")


if __name__ == "__main__":
    main_blockC_v6()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# ============================================================
# BlockD (H2O_v4_plot_vert_stdplev)
#
# Input:
#   H2O_vert_anom_special_stdplev.nc
#
# Plot time × pressure vertical sections (ppm),
# y-axis uses standard plev Pa (log, inverted).
# Significance:
#   /// for t-test, /// for bootstrap, xxx too ugly!!!!!!!!!!!!!!!!!!!!!
# ============================================================

import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

ROOT_H2O = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_H2O"
IN_C = os.path.join(ROOT_H2O, "H2O_vert_anom_special_stdplev.nc")

XTICKS = [0, 31, 61, 91, 122, 150, 181, 211, 242, 273, 304, 334]
XTICK_LABELS = ["Oct","Nov","Dec","Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep"]

def main_blockD_H2O_v4_plot_vert_stdplev():
    ds = xr.open_dataset(IN_C)
    years = ds.ref_year.values.astype(int)
    plev  = ds.plev.values
    anom  = ds.anom_ppm
    tmask = ds.tmask
    bmask = ds.bootmask

    n_days = anom.sizes["time_index"]
    x = np.arange(n_days)

    # sort by pressure ascending for plotting, then invert axis
    order = np.argsort(plev)
    plev_plot = plev[order]

    for i,y in enumerate(years):
        A = anom.isel(ref_year=i).values[:, order]   # (time, plev)
        T = tmask.isel(ref_year=i).values[:, order]
        B = bmask.isel(ref_year=i).values[:, order]

        fig, ax = plt.subplots(1,1, figsize=(7.2,4.8), constrained_layout=True)

        cf = ax.contourf(x, plev_plot, A.T, levels=np.linspace(-1,1,41), cmap="RdBu_r", extend="both")
        ax.contourf(x, plev_plot, T.T, levels=[0.5,1.5], hatches=["///"], colors="none")
        ax.contourf(x, plev_plot, B.T, levels=[0.5,1.5], hatches=["///"], colors="none")

        ax.set_yscale("log")
        ax.set_ylim(1e4, 10)
        ax.set_xticks(XTICKS); ax.set_xticklabels(XTICK_LABELS)
        ax.set_xlabel("Seasonal months (Oct–Sep)")
        ax.set_ylabel("Pressure [Pa]")
        ax.set_title(f"H2O anomaly vs climatology (ppm) — Year {y:04d}")

        cbar = fig.colorbar(cf, ax=ax, pad=0.02)
        cbar.set_label("H2O anomaly [ppm]")

        out_png = os.path.join(ROOT_H2O, f"H2O_vert_anom_stdplev_{y:04d}.png")
        out_pdf = os.path.join(ROOT_H2O, f"H2O_vert_anom_stdplev_{y:04d}.pdf")
        plt.savefig(out_png, dpi=300)
        plt.savefig(out_pdf)
        plt.close(fig)
        print("[BlockD_H2O_v4] Saved:", out_png)

    ds.close()
    print("[BlockD_H2O_v4] Done.")

if __name__ == "__main__":
    main_blockD_H2O_v4_plot_vert_stdplev()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# ============================================================
# BlockD2 (H2O_v4_plot_vert_stdplev_sig_split)  — UPDATED
#
# STRICT RULES:
#   - Base panel MUST be identical to your BlockD:
#       * levels = np.linspace(-1,1,41)
#       * cmap   = "RdBu_r"
#       * yscale log + ylim(1e4,10)
#   - DO NOT overlay t-test and bootstrap in same panel.
#   - Two outputs per year:
#       (1) t-test NON-significant hatched
#       (2) bootstrap NON-significant hatched
#   - Legend must state non-significant regions ONLY.
#   - REMOVE any "significant" legend items.
# ============================================================

import os
import numpy as np
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

ROOT_H2O = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_H2O"
IN_C = os.path.join(ROOT_H2O, "H2O_vert_anom_special_stdplev.nc")

XTICKS = [0, 31, 61, 91, 122, 150, 181, 211, 242, 273, 304, 334]
XTICK_LABELS = ["Oct","Nov","Dec","Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep"]

def _set_pub_style():
    """Only cosmetic style. Does NOT touch scientific plotting choices."""
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "DejaVu Sans"],
        "font.size": 9,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.linewidth": 0.8,
        "xtick.direction": "out",
        "ytick.direction": "out",
        "xtick.major.width": 0.8,
        "ytick.major.width": 0.8,
        "xtick.major.size": 3,
        "ytick.major.size": 3,
    })

def _base_panel(ax, x, plev_plot, A_T):
    """
    KEEP IDENTICAL to your current BlockD.
    Returns contourf handle for colorbar.
    """
    cf = ax.contourf(
        x, plev_plot, A_T,
        levels=np.linspace(-1, 1, 41),
        cmap="RdBu_r",
        extend="both"
    )

    ax.set_yscale("log")
    ax.set_ylim(1e4, 10)  # identical ylim
    ax.set_xticks(XTICKS)
    ax.set_xticklabels(XTICK_LABELS)
    ax.set_xlabel("Seasonal months (Oct–Sep)")
    ax.set_ylabel("Pressure [Pa]")
    return cf

def _overlay_sig(ax, x, plev_plot, sig_mask_T, hatch="///", zorder=10):
    """
    Overlay hatched region.
    sig_mask_T: (plev, time) float (0/1)
    """
    ax.contourf(
        x, plev_plot, sig_mask_T,
        levels=[0.5, 1.5],
        hatches=[hatch],
        colors="none",
        linewidths=0.0,   # no hatch border lines
        zorder=zorder
    )

def main_blockD2_H2O_v4_plot_vert_stdplev_sig_split():
    _set_pub_style()
    ds = xr.open_dataset(IN_C)

    years = ds.ref_year.values.astype(int)
    plev  = ds.plev.values
    anom  = ds.anom_ppm
    tmask = ds.tmask
    bmask = ds.bootmask

    n_days = anom.sizes["time_index"]
    x = np.arange(n_days)

    order = np.argsort(plev)
    plev_plot = plev[order]

    for i, y in enumerate(years):
        A = anom.isel(ref_year=i).values[:, order]  # (time, plev)
        T = tmask.isel(ref_year=i).values[:, order] # bool, True=significant
        B = bmask.isel(ref_year=i).values[:, order] # bool, True=significant

        # -------------------------
        # (1) t-test: hatch NON-significant
        # -------------------------
        fig, ax = plt.subplots(1, 1, figsize=(7.2, 4.8), constrained_layout=True)

        cf = _base_panel(ax, x, plev_plot, A.T)

        # invert mask: True -> NON-significant
        T_ns = (~T).astype(float).T  # (plev, time)
        _overlay_sig(ax, x, plev_plot, T_ns, hatch="///", zorder=10)

        ax.set_title(f"H2O anomaly vs climatology (ppm) — Year {y:04d} — t-test")

        cbar = fig.colorbar(cf, ax=ax, pad=0.02)
        cbar.set_label("H2O anomaly [ppm]")

        # legend: ONLY non-significant statement
        h_ns  = Patch(facecolor="none", edgecolor="0.2", hatch="///",
                      label="Hatched: non-significant (t-test)")
        ax.legend(handles=[h_ns],
                  loc="upper right", frameon=False, fontsize=8,
                  handlelength=2.0, handletextpad=0.6)

        out_png = os.path.join(ROOT_H2O, f"H2O_vert_anom_stdplev_ttest_NONSIG_D2_{y:04d}.png")
        out_pdf = os.path.join(ROOT_H2O, f"H2O_vert_anom_stdplev_ttest_NONSIG_D2_{y:04d}.pdf")
        plt.savefig(out_png, dpi=400)
        plt.savefig(out_pdf)
        plt.close(fig)
        print("[BlockD2_H2O] Saved:", out_png)

        # -------------------------
        # (2) bootstrap: hatch NON-significant
        # -------------------------
        fig, ax = plt.subplots(1, 1, figsize=(7.2, 4.8), constrained_layout=True)

        cf = _base_panel(ax, x, plev_plot, A.T)

        # invert mask: True -> NON-significant
        B_ns = (~B).astype(float).T  # (plev, time)
        _overlay_sig(ax, x, plev_plot, B_ns, hatch="///", zorder=10)

        ax.set_title(f"H2O anomaly vs climatology (ppm) — Year {y:04d} — bootstrap")

        cbar = fig.colorbar(cf, ax=ax, pad=0.02)
        cbar.set_label("H2O anomaly [ppm]")

        h_ns  = Patch(facecolor="none", edgecolor="0.2", hatch="///",
                      label="Hatched: non-significant (bootstrap)")
        ax.legend(handles=[h_ns],
                  loc="upper right", frameon=False, fontsize=8,
                  handlelength=2.0, handletextpad=0.6)

        out_png = os.path.join(ROOT_H2O, f"H2O_vert_anom_stdplev_boot_NONSIG_D2_{y:04d}.png")
        out_pdf = os.path.join(ROOT_H2O, f"H2O_vert_anom_stdplev_boot_NONSIG_D2_{y:04d}.pdf")
        plt.savefig(out_png, dpi=400)
        plt.savefig(out_pdf)
        plt.close(fig)
        print("[BlockD2_H2O] Saved:", out_png)

    ds.close()
    print("[BlockD2_H2O] Done.")

if __name__ == "__main__":
    main_blockD2_H2O_v4_plot_vert_stdplev_sig_split()
